In [1]:
# LayerNorm
import torch
import torch.nn as nn
import math

class LayerNormFast(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # x: [..., dim] 最后一层归一化
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean)/torch.sqrt(var+self.eps)
        return self.gamma * x_norm + self.beta

In [2]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        """
        参数：
            normalized_shape：需要归一的维度
            eps：小常数
            elementwise_affine：是否使用科学系的gamma和beta
        """
        super(LayerNorm, self).__init__()
        if isinstance(normalized_shape,int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(normalized_shape))
        else:
            self.gamma = None
            self.beta = None
    
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape), \
            f"输入维度 {x.dim()} 小于归一化维度 {len(self.normalized_shape)}"
        dims = list(range(-len(self.normalized_shape),0)) # [-2,-1]
        mean = x.mean(dims, keepdim=True)
        var = x.var(dims, keepdim=True, unbiased=False)
        std = torch.sqrt(var + self.eps)
        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            x_out = self.gamma * x_normalized + self.beta
        else:
            x_out = x_normalized
        return x_out
    
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super(RMSNorm,self).__init__()
        if isinstance(normalized_shape,int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None
        
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-(len(self.normalized_shape)),0))
        rms = torch.sqrt(x.pow(2).mean(dims, keepdim=True) + self.eps)
        if self.elementwise_affine:
            return self.gamma * (x/rms)
        return x/rms
    
# 示例输入
x = torch.randn(2,3,4)
layer_norm = LayerNorm(x.shape[-1])
rms_norm = RMSNorm(x.shape[-1])
out = layer_norm(x)
print(x)
print(out)
print(rms_norm(x))

tensor([[[-0.3837,  1.4554,  0.4561, -0.8562],
         [ 0.6142, -1.5578,  0.8507, -0.3781],
         [ 0.3791, -0.7226, -1.1410, -1.3359]],

        [[ 0.9136, -0.3118,  0.8885,  0.9752],
         [-1.9619, -2.2943,  1.9423,  0.4023],
         [-0.4022,  0.4103, -0.8498, -1.1665]]])
tensor([[[-0.6272,  1.4639,  0.3277, -1.1645],
         [ 0.7699, -1.5148,  1.0187, -0.2738],
         [ 1.6328, -0.0264, -0.6564, -0.9500]],

        [[ 0.5538, -1.7290,  0.5069,  0.6684],
         [-0.8520, -1.0429,  1.3896,  0.5054],
         [ 0.1685,  1.5395, -0.5868, -1.1212]]], grad_fn=<AddBackward0>)
tensor([[[-0.4285,  1.6255,  0.5095, -0.9563],
         [ 0.6411, -1.6262,  0.8880, -0.3947],
         [ 0.3914, -0.7461, -1.1780, -1.3793]],

        [[ 1.1178, -0.3815,  1.0870,  1.1931],
         [-1.0863, -1.2704,  1.0754,  0.2227],
         [-0.5178,  0.5282, -1.0941, -1.5019]]], grad_fn=<MulBackward0>)


In [3]:
# layernorm
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        """
        参数：
            normalized_shape：需要归一的维度
            eps：小常熟
            elementwise_affine：是否使用科学系的gamma和beta
        """
        super(LayerNorm, self).__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine

        if self.elementwise_affine:
            self.gemma = nn.Parameter(torch.ones(normalized_shape))
            self.beta = nn.Parameter(torch.zeros(normalized_shape))
        else:
            # self.register_parameter('gamma', None)
            # self.register_parameter('beta', None)
            self.gemma = None
            self.beta = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape), \
            f"输入维度 {x.dim()} 小于归一化维度 {len(self.normalized_shape)}"
        original_shape = x.shape

        dims = list(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        std = torch.sqrt(var + self.eps)

        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            x_out = self.gemma * x_normalized + self.beta
        else:
            x_out = x_normalized
        return x_out

# 示例输入
x = torch.randn(2,3,4)
layer_norm = LayerNorm(x.shape[-1])
out = layer_norm(x)
print(x)
print(out)

tensor([[[ 0.6816,  0.9945, -1.2394, -0.5350],
         [-0.9441, -0.5525, -0.8635, -0.7715],
         [-0.6843,  0.9478, -0.9146,  1.4627]],

        [[ 1.3226, -1.7417,  1.0371, -0.5217],
         [-1.1375,  1.3119, -1.9889,  1.4656],
         [ 0.1458,  0.0150, -0.4257,  1.1143]]])
tensor([[[ 0.7806,  1.1265, -1.3428, -0.5643],
         [-1.1011,  1.5737, -0.5505,  0.0779],
         [-0.8681,  0.7289, -1.0934,  1.2327]],

        [[ 1.0491, -1.4266,  0.8184, -0.4409],
         [-0.6968,  0.9282, -1.2616,  1.0302],
         [-0.1184, -0.3511, -1.1350,  1.6045]]], grad_fn=<AddBackward0>)


In [ ]:
# 多头注意力机制（selfattention，支持多头）
import torch
import torch.nn as nn
import math

class SelfAttentionFast(nn.Module):
    def __init__(self, d_model, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k =  d_model//n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask = None):
        B, L, D = x.shape
        # 分别表征 批次大小，序列长度，特征维度
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1,2) # shape: B, n_heads, L, d_k
        K = self.W_k(x).view(B, L, self.n_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B, L, self.n_heads, self.d_k).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k) # shape: B, n_heads, L, L
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        # matmul(attn, V) shape: B, n_heads, L, d_k
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, L, D)  # shape: B, L, n_heads, d_k
        return self.W_o(out)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads=8, dropout=0.1, bias=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model//n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        self.W_k = nn.Linear(d_model, d_model, bias=bias)
        self.W_v = nn.Linear(d_model, d_model, bias=bias)
        
        self.W_o = nn.Linear(d_model, d_model, bias=bias)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, is_causal=False):
        # mask: [batch_size, 1, seq_len, seq_len] 或 [batch_size, seq_len, seq_len] 0表示 mask，1 表示保留
        # is_causal: 是否使用因果掩码（上三角为0）

        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1,2)
        
        scores = torch.matmul(Q, K.transpose(-2,-1))//math.sqrt(self.d_k)

        if is_causal:
            causal_mask = torch.triu(
                torch.ones(seq_len, seq_len, device=x.device),
                diagonal=1
            ).bool()
            scores = scores.masked_fill(causal_mask, float('-inf'))

        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask==0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = torch.matmul(attn_weights, V)
        out = out.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(out)

        return output, attn_weights

In [ ]:
# 带有KV Cache的MHA
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_models, n_heads = 8, dropout=0.1, bias=True):
        super().__init__()
        assert d_models % n_heads == 0
        self.d_k = d_models//n_heads
        self.n_heads = n_heads
        
        self.W_q = nn.Linear(d_models, d_models, bias=bias)
        self.W_k = nn.Linear(d_models, d_models, bias=bias)
        self.W_v = nn.Linear(d_models, d_models, bias=bias)

        self.W_o = nn.Linear(d_models, d_models, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_key_value=None, use_cache=False, mask=None):
        B, L, D = x.shape

        Q = self.W_q(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2) # batch, n_heads, past_len+seq_len, d_k
            v = torch.cat([past_v, v], dim=2)
        present_key_value = (k, v) if use_cache else None

        score = torch.matmul(Q, K.transpose(-2,-1))/math.sqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask==0, float('-inf'))

        attn = F.softmax(score, dim=-1)
        attn = self.dropout(attn)

        output = torch.matmul(attn, V)
        output = output.transpose(1,2).contiguous().view(B,L,D)

        output = self.W_o(output)
        return output, present_key_value

In [ ]:
# FlashAttention
import torch
import torch.nn.functional as F
import math
from typing import Optional, Tuple

class FlashAttentionV1:
    def __init__(self, Br: int=128, Bc: int=128):
        self.Br = Br # Query Block size, Block size of Row 表示遍历Query块的行数
        self.Bc = Bc # Key/ Value size, Block size of Column, 表示遍历key/value块的列数

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, causal:bool=False, scale: Optional[float] = None)-> torch.Tensor:
        B, H, N, D = Q.shape # Batch size, Heads/ n_heads 注意力头数（已经是多头了）, Number of tokens/Sequence Length 序列长度, Dimension/d_head 每一个头的维度
        if scale is None:
            scale = 1.0 / math.sqrt(D)

        O = torch.zeros_like(Q)
        Tr = (N + self.Br - 1) // self.Br # Number of Tile Rows query方向的分块数，Tile划分
        Tc = (N + self.Bc - 1) // self.Bc # Number of Tile Columns key/value方向的分块数， Tile划分

        # V1版本：外层K，V块的循环，串行加载到SRAM
        for j in range(Tc):
            # 加载Kj，Vj到SRAM
            kv_start = j*self.Bc
            kv_end = min(kv_start+self.Bc, N)
            Kj = K[:, :, kv_start:kv_end, :]
            Vj = V[:, :, kv_start:kv_end, :]

            # 内层循环Q块，每个Q块和当前的Kj Vj计算
            for i in range(Tr):
                q_start = i * self.Br
                q_end = min(q_start + self.Br, N)
                Qi = Q[:, :, q_start:q_end, :]

                # 加载Oi，mi，li到SRAM
                Oi = Q[:, :, q_start:q_end, :].clone()
                mi = torch.full((B, H, q_end-q_start), float('-inf'), device=Q.device, dtype=torch.float32)
                li = torch.zeros(B, H, q_end - q_start, device=Q.device, dtype=torch.float32)

                # 计算Sij = Qi @ Kj^T
                Sij = torch.matmul(Qi, Kj.transpose(-2,-1)) * scale

                # 因果掩码
                if causal:
                    q_pos = torch.arange(q_start, q_end, device=Q.device).view(1,1,-1,1)
                    k_pos = torch.arange(kv_start, kv_end, device=Q.device).view(1,1,1,-1)
                    mask = q_pos < k_pos
                    Sij = Sij.masked_fill(mask, float('-inf'))
                
                # Online Softmax
                mij = Sij.max(dim=-1).values
                mij_new = torch.maximum(mi, mij)

                # 缩放并更新
                Pij = torch.exp(Sij - mij_new.unsqueeze(-1))
                Pij = torch.nan_to_num(Pij, nan=0.0)

                lij = Pij.sum(dim=-1)
                li_new = torch.exp(mi - mij_new) * li + lij

                # 更新输出
                Oi = (torch.exp(mi-mij_new).unsqueeze(-1) * Oi + torch.matmul(Pij, Vj))

                # 写回HBM
                Q[:, :, q_start:q_end, :] = Oi / li_new.unsqueeze(-1)

                mi = mij_new
                li = li_new
        return O
    
class FlashAttentionV2:
    def __inif__(self, Br:int=128, Bc:int=128):
        self.Br = Br
        self.Bc = Bc
    def forward(self, Q, K, V, causal: bool=False, scale:Optional[float]=None) -> torch.Tensor:
        B,H,N,D = Q.shape
        if scale is None:
            scale = 1.0/math.sqrt(D)

        O = torch.zeros_like(Q)
        Tr = (N + self.Br - 1) // self.Br
        Tc = (N + self.Bc - 1) // self.Bc

        for i in range(Tr):
            q_start = i*self.Br
            q_end = min(q_start+self.Br, N)
            Qi = Q[:, :, q_start:q_end, :]
            actual_Br = q_end - q_start

            mi = torch.full((B,H,actual_Br), float('-inf'), device=Q.device, dtype=torch.float32)
            li = torch.zeros(B, H, actual_Br, device=Q.device, dtype=torch.float32)
            Oi = torch.zeros(B, H, actual_Br, D, device=Q.device, dtype=Q.dtype)

            for j in range(Tc):
                kv_start = j * self.Bc
                kv_end = min(kv_start + self.Bc, N)
                Kj = K[:, :, kv_start:kv_end, :]
                Vj = V[:, :, kv_start:kv_end, :]
                
                # 计算 Sij
                Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale
                
                # 因果掩码（V2优化：提前跳过无效块）
                if causal:
                    q_pos = torch.arange(q_start, q_end, device=Q.device).view(1, 1, -1, 1)
                    k_pos = torch.arange(kv_start, kv_end, device=Q.device).view(1, 1, 1, -1)
                    
                    # V2优化：如果整个块都在因果掩码之下，直接跳过
                    if kv_end <= q_start:
                        continue  # 整个块都是未来信息，跳过
                    
                    mask = q_pos < k_pos
                    Sij = Sij.masked_fill(mask, float('-inf'))
                
                # Online Softmax（与V1相同数学，但顺序不同）
                mij = Sij.max(dim=-1).values
                mij_new = torch.maximum(mi, mij)
                
                Pij = torch.exp(Sij - mij_new.unsqueeze(-1))
                Pij = torch.nan_to_num(Pij, nan=0.0)
                
                lij = Pij.sum(dim=-1)
                li_new = torch.exp(mi - mij_new) * li + lij
                
                # 更新输出（在SRAM中完成，最后写回一次！）
                Oi = (torch.exp(mi - mij_new).unsqueeze(-1) * Oi + 
                      torch.matmul(Pij, Vj))
                
                mi = mij_new
                li = li_new
            
            # V2关键：每个Q块只写回一次HBM！
            O[:, :, q_start:q_end, :] = Oi / li.unsqueeze(-1)
        
        return O


class FlashAttentionV2Optimized(FlashAttentionV2):
    """
    FlashAttention V2 的进一步优化版本
    
    额外优化：
    1. 因果掩码的块级跳过（V2核心优化）
    2. Split-K for long sequences
    3. 更好的warp级并行
    """
    
    def forward(self, Q, K, V, causal=False, scale=None):
        B, H, N, D = Q.shape
        if scale is None:
            scale = 1.0 / math.sqrt(D)
        
        O = torch.zeros_like(Q)
        Tr = (N + self.Br - 1) // self.Br
        Tc = (N + self.Bc - 1) // self.Bc
        
        for i in range(Tr):
            q_start = i * self.Br
            q_end = min(q_start + self.Br, N)
            Qi = Q[:, :, q_start:q_end, :]
            actual_Br = q_end - q_start
            
            mi = torch.full((B, H, actual_Br), float('-inf'), 
                           device=Q.device, dtype=torch.float32)
            li = torch.zeros(B, H, actual_Br, device=Q.device, dtype=torch.float32)
            Oi = torch.zeros(B, H, actual_Br, D, device=Q.device, dtype=Q.dtype)
            
            # V2关键优化：确定K,V的有效范围
            # 对于causal，只需要处理 k <= q 的块
            kv_start_j = 0 if not causal else max(0, q_start // self.Bc)
            
            for j in range(kv_start_j, Tc):
                kv_start = j * self.Bc
                kv_end = min(kv_start + self.Bc, N)
                Kj = K[:, :, kv_start:kv_end, :]
                Vj = V[:, :, kv_start:kv_end, :]
                
                Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale
                
                if causal:
                    q_pos = torch.arange(q_start, q_end, device=Q.device).view(1, 1, -1, 1)
                    k_pos = torch.arange(kv_start, kv_end, device=Q.device).view(1, 1, 1, -1)
                    mask = q_pos < k_pos
                    Sij = Sij.masked_fill(mask, float('-inf'))
                
                # Online softmax
                mij = Sij.max(dim=-1).values
                mij_new = torch.maximum(mi, mij)
                
                Pij = torch.exp(Sij - mij_new.unsqueeze(-1))
                Pij = torch.nan_to_num(Pij, nan=0.0)
                
                lij = Pij.sum(dim=-1)
                li_new = torch.exp(mi - mij_new) * li + lij
                
                Oi = (torch.exp(mi - mij_new).unsqueeze(-1) * Oi + 
                      torch.matmul(Pij, Vj))
                
                mi = mij_new
                li = li_new
            
            O[:, :, q_start:q_end, :] = Oi / li.unsqueeze(-1)
        
        return O